Script for building a FROZEN dataset, i.e. the training/test split is performed here and exported so that the exact same split is used by ALL downstream models.

[Runtime: Trivial]

--------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, subprocess
import json
import pandas as pd
import numpy as np
import re
from datetime import datetime, timezone

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
SUBSET       = config['subset']
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']

# Wire in RANDOM_SEED determinism (if valid int):
if isinstance(RANDOM_SEED, (int, np.integer)):
    np.random.seed(int(RANDOM_SEED))
    print(f"[INFO] RANDOM_SEED set to {int(RANDOM_SEED)} (deterministic behavior enabled).")
else:
    RANDOM_SEED = None
    print("[INFO] RANDOM_SEED is not a valid int; proceeding with non-deterministic randomness.")

OVERWRITE_DATASET = config['ML_prep']['overwrite_data_prep']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

# For file targeting:
ATLAS_FAMILY =  config['parcellation']['atlas']                     # STR
NUM_ROIS      = config['parcellation']['n_rois']                    # INT

EPOCH_LENGTH  = config['compute_correlations']['epoch_length']      # INT
EPOCH_OVERLAP = config['compute_correlations']['epoch_overlap']     # INT

USE_MANIFEST_FILTER = config['ML_prep']['use_manifest_filter']      # BOOL   <-- if True, only collects data from subject_IDs & session_IDs explicitly mentioned in 'MEG_runs'

# ML preprocessing parameters:
ML_CENTRALITY_TYPES = config['ML_prep'].get('ML_centrality_types')
if not ML_CENTRALITY_TYPES:  # treats 'None', 'null' and '[]' as "use previous/upstream settings"
    ML_CENTRALITY_TYPES = config["compute_correlations"]["centrality_types"]
ML_GRAPH_METRICS = config['ML_prep'].get('ML_graph_metrics')
if not ML_GRAPH_METRICS:
    ML_GRAPH_METRICS = config["compute_correlations"]["graph_metrics"]
ROI_METRIC_NAMES = [f"{str(centrality_type).lower()}Centrality" for centrality_type in ML_CENTRALITY_TYPES]
GLOBAL_METRIC_NAMES = [str(metric_name) for metric_name in ML_GRAPH_METRICS]
ALLOWED_METRIC_NAMES = set(ROI_METRIC_NAMES) | set(GLOBAL_METRIC_NAMES)

STANDARDIZE = config['ML_preprocessing']['standardize']
CENTER_ONLY = config['ML_preprocessing']['center_only']
HANDLE_MISSING = config['ML_preprocessing']['handle_missing']

USE_HOLDOUT = config['ML_preprocessing']['use_holdout']
HOLDOUT_FRACTION = config['ML_preprocessing']['holdout_fraction']

SPLIT_BY_SUBJECT = config['ML_preprocessing']['split_by_subject']
if not SPLIT_BY_SUBJECT:
    print("[WARNING:] 'split_by_subject' = 'False': train/test split may place the same subject in both sets (potential data leakage); use only if you fully understand the implications.")

ENFORCE_CLASS_BALANCE = config['ML_preprocessing']['enforce_class_balance']


# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
MEG_PARAMETERS_PATH  = BASE_DIRECTORY / 'MEG_manifest.csv'

### INPUTS:

DATA_DIR = Path(config['ML_prep']['input_data_directory'])

### OUTPUTS:

OUTPUT_DIR = Path(BASE_DIRECTORY) / config['ML_prep']['training_data_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# __________________________________________________________________________________________________________
### INITIALIZATION:

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
MEG_runs = pd.read_csv(MEG_PARAMETERS_PATH)
if "MEG_session_ID" in MEG_runs.columns:
    MEG_runs = MEG_runs.rename(columns={"MEG_session_ID": "session_ID"})

In [ ]:
# =====================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# =====================================================================

# FILTERING (if enabled):
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to MEG_runs...")
    print(f"[FILTER] Starting with {len(MEG_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(MEG_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = MEG_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        MEG_runs = MEG_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(MEG_runs)
        mask = MEG_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        MEG_runs = MEG_runsloc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(MEG_runs)
        mask = MEG_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        MEG_runs = MEG_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(MEG_runs):,} rows.\n")

# DIAGNOSTIC SUBSETTING (if enabled):
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    MEG_runs = MEG_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    MEG_runs = MEG_runs.reset_index(drop=True)
    display(MEG_runs)

--------

In [ ]:
# __________________________________________________________________________________________________________
### BUILD MASTER HMM DATAFRAME FROM EPOCHED GRAPH METRICS (tidy timecourses)
#
# Input assumptions:
#   DATA_DIR = root directory under which we recursively search for files matching:
#       *epoched_window-<EPOCH_LENGTH>_overlap-<EPOCH_OVERLAP>_timecourses.csv
#
# For each subject/session we expect ONE timecourses file:
#   <subject_ID>_<session_ID>_epoched_window-<EPOCH_LENGTH>_overlap-<EPOCH_OVERLAP>_timecourses.csv
#
# Structure of each timecourses CSV (tidy):
#   columns:
#       subject_ID   (str)
#       session_ID   (str)
#       level        (str: 'roi' or 'global')
#       roi_label    (int for level='roi', NaN for level='global')
#       metric_name  (str, e.g. 'eigenvectorCentrality', 'degreeCentrality', 'Q')
#       time_index   (int, e.g. 1..N windows)
#       value        (float)
#
# We will:
#   - Restrict to subjects/sessions listed in MEG_runs if USE_MANIFEST_FILTER=True.
#   - Restrict to metric_name ∈ ALLOWED_METRIC_NAMES (derived from ML_* config).
#   - Pivot ROI-level rows into:
#         index   : time_index (int)
#         columns : MultiIndex (roi_label, metric_name)
#   - Pivot global rows (if any) into:
#         index   : time_index (int)
#         columns : metric_name
#   - Align time_indices between ROI and global pivots.
#   - Flatten columns into feature names:
#         ROIxxx_<metric_name>         for ROI-level metrics
#         GLOBAL_<metric_name>         for global metrics (e.g. Q)
#
# Output:
#   master_hmm_df: long-format DataFrame with columns:
#       ['subject_ID', 'session_ID', 'time_index', <feature_columns>]
# __________________________________________________________________________________________________________

# Set column name for ROI labels in tidy timecourses:
ROI_LABEL_COL = "roi_label"

# Set filename pattern for (tidy) timecourses files:
CENTRALITY_FILENAME_RE = re.compile(
    rf"^(?P<subject_ID>[^_]+)_(?P<session_ID>[^_]+)_epoched_window-(?P<epoch_len>\d+)_overlap-(?P<epoch_ovlp>\d+)_timecourses\.csv$")

# Helper function: error/warning handling respecting HARD_STOP:
def handle_error(message: str):
    if HARD_STOP:
        raise RuntimeError(message)
    else:
        print(f"[WARN] {message}")

# Optional: build (subject_ID, session_ID) filter from 'MEG_runs'
if USE_MANIFEST_FILTER:
    allowed_pairs = set(
        zip(
            MEG_runs["subject_ID"].astype(str),
            MEG_runs["session_ID"].astype(str)))
    print(f"[INFO] MEG_runs filter enabled: {len(allowed_pairs)} (subject_ID, session_ID) pairs.")
else:
    allowed_pairs = None
    print("[INFO] MEG_runs filter DISABLED: all discovered files will be considered.")

# Main -- recursively scan DATA_DIR, build master HMM dataframe:
def build_master_hmm_dataframe_from_dir(data_dir: Path) -> pd.DataFrame:
    # Pattern for tidy timecourses files (search recursively)
    timecourses_pattern = f"*epoched_window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}_timecourses.csv"

    print(f"\n[INFO] Recursively searching for timecourses files under: {data_dir}")
    print(f"[INFO] EPOCH_LENGTH={EPOCH_LENGTH}, EPOCH_OVERLAP={EPOCH_OVERLAP}")
    timecourses_paths = sorted(data_dir.rglob(timecourses_pattern))
    print(f"[INFO] Found {len(timecourses_paths)} timecourses files matching pattern '{timecourses_pattern}'.")

    all_rows = []

    for timecourses_path in timecourses_paths:
        filename = timecourses_path.name
        match = CENTRALITY_FILENAME_RE.match(filename)
        if not match:
            handle_error(f"Skipping file with unexpected timecourses filename format: {filename}")
            continue

        subject_ID = match.group("subject_ID")
        session_ID = match.group("session_ID")
        epoch_len  = int(match.group("epoch_len"))
        epoch_ovlp = int(match.group("epoch_ovlp"))

        # Sanity-check of epoching parameters:
        if epoch_len != EPOCH_LENGTH or epoch_ovlp != EPOCH_OVERLAP:
            handle_error(
                f"{filename}: epoch settings in filename (length={epoch_len}, overlap={epoch_ovlp}) "
                f"do not match current config (length={EPOCH_LENGTH}, overlap={EPOCH_OVERLAP}). Skipping.")
            continue

        # Optional filter using MEG_runs
        if allowed_pairs is not None and (subject_ID, session_ID) not in allowed_pairs:
            continue

        parent_dir = timecourses_path.parent

        # Load tidy timecourses CSV
        try:
            timecourses_df = pd.read_csv(timecourses_path)
        except Exception as exc:
            handle_error(f"Failed to read {timecourses_path}: {exc}")
            continue

        required_cols = [
            "subject_ID",
            "session_ID",
            "level",
            ROI_LABEL_COL,
            "metric_name",
            "time_index",
            "value"]
        missing_cols = [column_name for column_name in required_cols if column_name not in timecourses_df.columns]
        if missing_cols:
            handle_error(f"{filename}: missing required columns: {missing_cols}")
            continue

        # Sanity-check to ensure subject_ID / session_ID is in the file:
        subj_vals = timecourses_df["subject_ID"].astype(str).unique()
        sess_vals = timecourses_df["session_ID"].astype(str).unique()
        if len(subj_vals) != 1 or subj_vals[0] != subject_ID:
            handle_error(
                f"{filename}: subject_ID in file {subj_vals} does not match expected '{subject_ID}'.")
        if len(sess_vals) != 1 or sess_vals[0] != session_ID:
            handle_error(
                f"{filename}: session_ID in file {sess_vals} does not match expected '{session_ID}'.")

        # Filter by metric_name (using 'ALLOWED_METRIC_NAMES'):
        original_row_count = timecourses_df.shape[0]
        timecourses_df = timecourses_df[timecourses_df["metric_name"].isin(ALLOWED_METRIC_NAMES)].copy()

        if timecourses_df.empty:
            handle_error(
                f"{filename}: no rows remain after filtering by allowed metric_name values "
                f"{sorted(ALLOWED_METRIC_NAMES)}; skipping this subject/session.")
            continue

        if timecourses_df.shape[0] < original_row_count:
            print(
                f"[INFO] {filename}: filtered {original_row_count - timecourses_df.shape[0]} row(s) "
                f"based on ML metric selection.")

        # Split into ROI-level and global rows:
        timecourses_df["level"] = timecourses_df["level"].astype(str)
        roi_rows = timecourses_df[timecourses_df["level"] == "roi"].copy()
        global_rows = timecourses_df[timecourses_df["level"] == "global"].copy()

        if roi_rows.empty:
            handle_error(f"{filename}: no ROI-level rows (level == 'roi') remain after filtering; skipping.")
            continue

        # ROI sanity-check:
        roi_labels = roi_rows[ROI_LABEL_COL].dropna().unique()
        n_roi_found = len(roi_labels)
        if n_roi_found != NUM_ROIS:
            handle_error(
                f"{filename}: expected {NUM_ROIS} ROIs, found {n_roi_found} "
                f"(unique {ROI_LABEL_COL} values).")

        # Ensure 'time_index' is integer:
        roi_rows["time_index"] = roi_rows["time_index"].astype(int)

        # ------------------------------------------------------------------
        # Pivot ROI-level data:
        #   index   : time_index
        #   columns : (roi_label, metric_name)
        # ------------------------------------------------------------------
        roi_pivot = roi_rows.pivot_table(
            index="time_index",
            columns=[ROI_LABEL_COL, "metric_name"],
            values="value")
        roi_pivot = roi_pivot.sort_index(axis=0).sort_index(axis=1)

        # ------------------------------------------------------------------
        # Pivot global metrics (if any):
        #   index   : time_index
        #   columns : metric_name
        # ------------------------------------------------------------------
        if global_rows.empty:
            global_pivot = None
            has_global_metrics = False
        else:
            global_rows["time_index"] = global_rows["time_index"].astype(int)
            global_pivot = global_rows.pivot_table(
                index="time_index",
                columns="metric_name",
                values="value")
            global_pivot = global_pivot.sort_index(axis=0).sort_index(axis=1)
            has_global_metrics = True

        times_roi = roi_pivot.index.to_numpy()

        # ------------------------------------------------------------------
        # Align times across ROI & global metrics
        # ------------------------------------------------------------------
        if has_global_metrics:
            times_global = global_pivot.index.to_numpy()
            shared_times = np.intersect1d(times_roi, times_global)
            if len(shared_times) == 0:
                handle_error(
                    f"No overlapping timepoints between ROI and global metrics for "
                    f"{subject_ID}, {session_ID}. Skipping.")
                continue

            roi_aligned = roi_pivot.loc[shared_times]
            global_aligned = global_pivot.loc[shared_times]

            combined = pd.concat([roi_aligned, global_aligned], axis=1)
        else:
            shared_times = times_roi
            combined = roi_pivot.loc[shared_times]

        # ------------------------------------------------------------------
        # Flatten MultiIndex columns --> feature names
        # ------------------------------------------------------------------
        new_column_names = []
        for column in combined.columns:
            if isinstance(column, tuple):
                # ROI-level feature = '(roi_label, metric_name)':
                roi_label_value, metric_name = column
                try:
                    roi_int = int(roi_label_value)
                    roi_str = f"ROI{roi_int:03d}"
                except (TypeError, ValueError):
                    roi_str = f"ROI{roi_label_value}"
                new_name = f"{roi_str}_{metric_name}"
            else:
                # Global feature = 'metric_name':
                new_name = f"GLOBAL_{column}"
            new_column_names.append(new_name)

        combined.columns = new_column_names

        # Construct long-format rows with [subject_ID, session_ID, time_index]:
        combined = combined.reset_index().rename(columns={"time_index": "time_index"})
        if "time_index" not in combined.columns and "index" in combined.columns:
            combined = combined.rename(columns={"index": "time_index"})

        combined["subject_ID"] = subject_ID
        combined["session_ID"] = session_ID

        # Column order = IDs first, then feature columns (& sorted for stability):
        id_columns = ["subject_ID", "session_ID", "time_index"]
        feature_columns = [column for column in combined.columns if column not in id_columns]
        feature_columns = sorted(feature_columns)
        combined = combined[id_columns + feature_columns]

        all_rows.append(combined)

        print(
            f"[INFO] Loaded {subject_ID}, {session_ID} from {parent_dir}: "
            f"{combined.shape[0]} timepoints, {len(feature_columns)} features "
            f"({'ROI + global' if has_global_metrics else 'ROI only'}).")

    if not all_rows:
        handle_error("No valid timecourses files found after filtering. master_hmm_df will be empty.")
        master_df = pd.DataFrame(columns=["subject_ID", "session_ID", "time_index"])
    else:
        master_df = pd.concat(all_rows, ignore_index=True)
        master_df = master_df.sort_values(["subject_ID", "session_ID", "time_index"]).reset_index(drop=True)
        print(f"\n[INFO] master_hmm_df shape: {master_df.shape}")
        print("[INFO] Columns (first 10):", list(master_df.columns[:10]))

    return master_df

# ----------------------------------------------------------------------
# Main execution:
master_hmm_df = build_master_hmm_dataframe_from_dir(DATA_DIR)

print("\n[INFO] Preview of master_hmm_df:")
display(master_hmm_df.head(10))

In [ ]:
# __________________________________________________________________________________________________________
### ML PREP: clean, standardize, and prepare HMM dataset tables (NO PCA here)
#
# Input:
#   master_hmm_df with columns:
#       ['subject_ID', 'session_ID', 'time_index', <feature_columns>]
#
# Behaviour:
#   - Verifies feature consistency across (subject_ID, session_ID).
#   - Handles missing values according to HANDLE_MISSING:
#       * 'drop'         : drop any row with NaN/±inf in feature columns
#       * 'impute_zero'  : replace NaN/±inf with 0.0
#       * 'impute_mean'  : per-feature mean imputation (error or drop features if all-NaN)
#   - Applies centering / standardization:
#       * STANDARDIZE=True  → z-score (mean 0, std 1) across all rows per feature
#       * CENTER_ONLY=True  → subtract mean only (no scaling)
#       * otherwise         → leave values as-is
#   - Builds sequence descriptors per (subject_ID, session_ID).
#   - Prepares dataset directory name (but does NOT write files in this cell).
# __________________________________________________________________________________________________________

if master_hmm_df.empty:
    handle_error(
        "master_hmm_df is empty; no training matrix will be created. "
        "Check upstream epoching / metric selection.")
else:
    # Identify ID vs feature columns and check cross-subject consistency:
    id_columns = ["subject_ID", "session_ID", "time_index"]
    for required_column in id_columns:
        if required_column not in master_hmm_df.columns:
            handle_error(f"master_hmm_df is missing required ID column: '{required_column}'")

    feature_columns_reference = [
        column_name for column_name in master_hmm_df.columns
        if column_name not in id_columns]

    if not feature_columns_reference:
        handle_error("No feature columns found in master_hmm_df (only ID columns present).")

    feature_set_reference = set(feature_columns_reference)

    inconsistent_pairs = []
    grouped_by_subject_session = master_hmm_df.groupby(["subject_ID", "session_ID"], sort=False)

    for (subject_id, session_id), group_df in grouped_by_subject_session:
        current_feature_columns = [
            column_name for column_name in group_df.columns
            if column_name not in id_columns]
        current_feature_set = set(current_feature_columns)

        missing_features = feature_set_reference - current_feature_set
        extra_features   = current_feature_set - feature_set_reference

        if missing_features or extra_features:
            inconsistent_pairs.append({
                "subject_ID": subject_id,
                "session_ID": session_id,
                "missing_features": sorted(missing_features),
                "extra_features": sorted(extra_features)})

    if inconsistent_pairs:
        message_lines = [
            "[ERROR] Feature inconsistency detected across subject/session combinations."]
        for descriptor in inconsistent_pairs[:5]:
            message_lines.append(
                f"  - {descriptor['subject_ID']}, {descriptor['session_ID']}: "
                f"missing={descriptor['missing_features']}, extra={descriptor['extra_features']}")
        if len(inconsistent_pairs) > 5:
            message_lines.append(
                f"  ... and {len(inconsistent_pairs) - 5} more subject/session pairs.")
        message = "\n".join(message_lines)

        if HARD_STOP:
            raise RuntimeError(message)
        else:
            print(message)
            bad_pairs = {
                (descriptor["subject_ID"], descriptor["session_ID"])
                for descriptor in inconsistent_pairs}
            print(
                f"[WARN] Dropping {len(bad_pairs)} subject/session sequence(s) "
                f"due to feature inconsistency (HARD_STOP=False).")
            master_hmm_df = (
                master_hmm_df[
                    ~master_hmm_df.set_index(["subject_ID", "session_ID"]).index.isin(bad_pairs)].reset_index(drop=True))

    # Re-identify features in case we dropped anything above:
    feature_columns_reference = [column_name for column_name in master_hmm_df.columns if column_name not in id_columns]

    # Handle missing values (e.g. NaNs, ±inf) in feature columns:
    features_matrix = master_hmm_df[feature_columns_reference].copy()

    # Treat ±inf as missing value:
    features_matrix = features_matrix.replace([np.inf, -np.inf], np.nan)

    missing_mask_any = features_matrix.isna().any(axis=1)
    rows_with_missing = int(missing_mask_any.sum())

    if HANDLE_MISSING == "drop":
        if rows_with_missing > 0:
            print(f"[INFO] HANDLE_MISSING='drop': dropping {rows_with_missing} row(s) with missing values.")
        features_matrix = features_matrix[~missing_mask_any]
        master_hmm_df = master_hmm_df.loc[features_matrix.index].reset_index(drop=True)
        features_matrix = features_matrix.reset_index(drop=True)

        if features_matrix.empty:
            handle_error("After dropping rows with missing values, no data remains for training.")

    elif HANDLE_MISSING == "impute_zero":
        if rows_with_missing > 0:
            print(f"[INFO] HANDLE_MISSING='impute_zero': imputing {rows_with_missing} row(s) with 0.0.")
        features_matrix = features_matrix.fillna(0.0)

    elif HANDLE_MISSING == "impute_mean":
        if rows_with_missing > 0:
            print(f"[INFO] HANDLE_MISSING='impute_mean': imputing {rows_with_missing} row(s) with feature means.")
        feature_means = features_matrix.mean(axis=0, skipna=True)
        all_nan_features = feature_means[feature_means.isna()].index.tolist()

        if all_nan_features:
            message = (
                "[ERROR] The following feature(s) have all-NaN values across all rows: "
                f"{all_nan_features}")
            if HARD_STOP:
                raise RuntimeError(message)
            else:
                print(message)
                print("[WARN] Dropping all-NaN feature(s) (HARD_STOP=False).")
                features_matrix = features_matrix.drop(columns=all_nan_features)
                master_hmm_df = master_hmm_df.drop(columns=all_nan_features)
                feature_columns_reference = [
                    column_name for column_name in feature_columns_reference
                    if column_name not in all_nan_features]
                feature_means = feature_means.drop(index=all_nan_features)

        features_matrix = features_matrix.fillna(feature_means)

    else:
        handle_error(f"Unknown HANDLE_MISSING strategy: {HANDLE_MISSING}")

    # Centering / standardization:
    scaler_info = {
        "centered": False,
        "standardized": False,
        "feature_means": {},
        "feature_stds": {}}

    if STANDARDIZE:
        feature_means = features_matrix.mean(axis=0)
        feature_stds  = features_matrix.std(axis=0, ddof=0)

        zero_std_features = feature_stds[feature_stds == 0].index.tolist()
        if zero_std_features:
            print(
                "[INFO] Constant features detected (std=0); they will be mean-centered "
                f"but not scaled: {zero_std_features}")

        features_scaled = features_matrix - feature_means

        nonzero_std_mask = feature_stds > 0
        if nonzero_std_mask.any():
            features_scaled.loc[:, nonzero_std_mask] = (
                features_scaled.loc[:, nonzero_std_mask] / feature_stds[nonzero_std_mask])

        scaler_info["centered"] = True
        scaler_info["standardized"] = True
        scaler_info["feature_means"] = feature_means.to_dict()
        scaler_info["feature_stds"]  = feature_stds.to_dict()

    elif CENTER_ONLY:
        feature_means = features_matrix.mean(axis=0)
        features_scaled = features_matrix - feature_means

        scaler_info["centered"] = True
        scaler_info["standardized"] = False
        scaler_info["feature_means"] = feature_means.to_dict()
        scaler_info["feature_stds"]  = {column_name: 1.0 for column_name in features_matrix.columns}

    else:
        features_scaled = features_matrix.copy()
        scaler_info["centered"] = False
        scaler_info["standardized"] = False
        scaler_info["feature_means"] = {column_name: 0.0 for column_name in features_matrix.columns}
        scaler_info["feature_stds"]  = {column_name: 1.0 for column_name in features_matrix.columns}

    # Reattach ID columns to form the final dataset table:
    id_frame = master_hmm_df[id_columns].reset_index(drop=True)
    features_scaled = features_scaled.reset_index(drop=True)

    training_df = pd.concat([id_frame, features_scaled], axis=1)

    feature_columns_final = sorted([c for c in training_df.columns if c not in id_columns])
    training_df = training_df[id_columns + feature_columns_final]

    # Build sequence descriptors (per subject_ID x session_ID):
    sequence_descriptors = []
    grouped_for_sequences = training_df.groupby(["subject_ID", "session_ID"], sort=False)

    for (subject_id, session_id), sequence_df in grouped_for_sequences:
        row_indices = sequence_df.index.to_numpy()
        sequence_descriptors.append({
            "subject_ID": str(subject_id),
            "session_ID": str(session_id),
            "length": int(sequence_df.shape[0]),
            "row_start": int(row_indices.min()),
            "row_end": int(row_indices.max())})

    num_sessions = len(sequence_descriptors)

    # Derive 'num_samples' summary for dataset naming:
    if sequence_descriptors:
        lengths_arr = np.array([d["length"] for d in sequence_descriptors], dtype=int)
        unique_lengths = np.unique(lengths_arr)
        if unique_lengths.size == 1:
            num_samples_str = f"{int(unique_lengths[0])}"
        else:
            num_samples_str = f"{int(lengths_arr.min())}-{int(lengths_arr.max())}"
    else:
        num_samples_str = "0"

    # Calculate overlap percentage (& store as string 00-99):
    if EPOCH_LENGTH and EPOCH_LENGTH > 0 and EPOCH_OVERLAP is not None:
        overlap_fraction = float(EPOCH_OVERLAP) / float(EPOCH_LENGTH)
        overlap_percent = int(round(overlap_fraction * 100.0))
        overlap_percent = max(0, min(99, overlap_percent))
    else:
        overlap_percent = 0
    overlap_perc_str = f"{overlap_percent:02d}"

    # Set filename tag for preprocessing mode:
    if STANDARDIZE:
        tag_str = "STD"
    elif CENTER_ONLY:
        tag_str = "CNT"
    else:
        tag_str = "RAW"

    # Dataset directory name:
    dataset_dirname = (
        f"n-{num_sessions}_"
        f"{ATLAS_FAMILY}_"
        f"{NUM_ROIS}-ROIs_"
        f"t-{num_samples_str}({overlap_perc_str})_"
        f"{tag_str}")

    DATASET_DIR = OUTPUT_DIR / dataset_dirname

    print("\n[ML PREP] Dataset table prepared (in-memory).")
    print(f"[ML PREP] training_df shape: {training_df.shape}")
    print(f"[ML PREP] # sequences: {num_sessions}")
    print(f"[ML PREP] Dataset directory will be: {DATASET_DIR}")

In [ ]:
# __________________________________________________________________________________________________________
### FREEZE DATASET: build X_all / X_train / X_test & write to disk
#
# Uses:
#   - training_df          : [subject_ID, session_ID, time_index, <features>]
#   - sequence_descriptors : list of dicts with row_start / row_end per sequence
#   - config toggles: USE_HOLDOUT, HOLDOUT_FRACTION, RANDOM_SEED
#   - split controls: SPLIT_BY_SUBJECT, ENFORCE_CLASS_BALANCE
#   - overwrite: OVERWRITE_DATASET
#
# Outputs (on disk) in DATASET_DIR:
#   - X_all.csv
#   - X_train.csv
#   - X_test.csv
#   - provenance.json
#
# Also writes/updates a stable pointer file in OUTPUT_DIR:
#   - OUTPUT_DIR / "LATEST_DATASET.json"
#     (contains the full path to DATASET_DIR and related metadata)
# __________________________________________________________________________________________________________

if training_df.empty:
    handle_error("training_df is empty; cannot freeze dataset.")
else:
    # Overwrite guard (directory-level):
    if DATASET_DIR.exists() and not OVERWRITE_DATASET:
        raise RuntimeError(
            f"[FREEZE ERROR] Dataset directory already exists and OVERWRITE_DATASET=False:\n  {DATASET_DIR}")

    if DATASET_DIR.exists() and OVERWRITE_DATASET:
        print(f"[FREEZE] OVERWRITE_DATASET=True --> will overwrite files in: {DATASET_DIR}")

    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    # Build sequence table
    sequences_df = pd.DataFrame(sequence_descriptors)
    if sequences_df.empty:
        handle_error("sequence_descriptors is empty; cannot freeze dataset.")
    sequences_df["subject_ID"] = sequences_df["subject_ID"].astype(str)
    sequences_df["session_ID"] = sequences_df["session_ID"].astype(str)
    sequences_df["length"] = sequences_df["length"].astype(int)
    sequences_df["row_start"] = sequences_df["row_start"].astype(int)
    sequences_df["row_end"] = sequences_df["row_end"].astype(int)
    sequences_df = sequences_df.reset_index(drop=True)
    sequences_df["sequence_index"] = sequences_df.index

    # Attach group_ID labels (if available) for optional stratification:
    sequences_df["group_ID"] = np.nan
    if "group_ID" in MEG_runs.columns:
        manifest_labels = MEG_runs[["subject_ID", "session_ID", "group_ID"]].copy()
        manifest_labels["subject_ID"] = manifest_labels["subject_ID"].astype(str)
        manifest_labels["session_ID"] = manifest_labels["session_ID"].astype(str)

        sequences_df = sequences_df.merge(
            manifest_labels,
            on=["subject_ID", "session_ID"],
            how="left",
            suffixes=("", "_manifest"))
        # If merge created group_ID_manifest due to existing group_ID, prefer the manifest's values:
        if "group_ID_manifest" in sequences_df.columns:
            sequences_df["group_ID"] = sequences_df["group_ID_manifest"]
            sequences_df = sequences_df.drop(columns=["group_ID_manifest"])
        # Ensure 'group_ID' is str wherever present:
        if "group_ID" in sequences_df.columns:
            sequences_df["group_ID"] = sequences_df["group_ID"].astype(str)

    # Random seed handling:
    seed_is_int = isinstance(RANDOM_SEED, (int, np.integer))
    if seed_is_int:
        rng = np.random.RandomState(int(RANDOM_SEED))
        random_seed_effective = int(RANDOM_SEED)
        print(f"[FREEZE] Using deterministic RANDOM_SEED={random_seed_effective}")
    else:
        rng = np.random.RandomState(None)
        random_seed_effective = None
        print("[FREEZE] RANDOM_SEED is not an int --> using non-deterministic randomness")

    # Helper: build table from selected sequences:
    id_columns = ["subject_ID", "session_ID", "time_index"]
    feature_columns_final = [c for c in training_df.columns if c not in id_columns]
    feature_columns_final = sorted(feature_columns_final)

    def build_table_from_sequences(selected_sequence_indices: np.ndarray) -> pd.DataFrame:
        selected_sequence_table = sequences_df.loc[selected_sequence_indices].copy()

        chunks = []
        for _, seq_row in selected_sequence_table.iterrows():
            row_start = int(seq_row["row_start"])
            row_end = int(seq_row["row_end"])
            seq_slice = training_df.iloc[row_start : row_end + 1, :]
            chunks.append(seq_slice)

        if chunks:
            out_df = pd.concat(chunks, axis=0, ignore_index=True)
        else:
            out_df = training_df.iloc[0:0, :].copy()  # empty with same columns

        return out_df, selected_sequence_table

    # Optional class-balance enforcement checks:
    if ENFORCE_CLASS_BALANCE:
        if "group_ID" not in MEG_runs.columns:
            raise RuntimeError("[FREEZE ERROR] ENFORCE_CLASS_BALANCE=True but MEG_runs has no 'group_ID' column.")

        if "group_ID" not in sequences_df.columns:
            raise RuntimeError("[FREEZE ERROR] ENFORCE_CLASS_BALANCE=True but sequence labels could not be constructed.")

        bad_mask = sequences_df["group_ID"].isna() | (sequences_df["group_ID"].astype(str) == "UNKNOWN")
        if bad_mask.any():
            bad_examples = sequences_df.loc[bad_mask, ["subject_ID", "session_ID", "group_ID"]].head(10)
            raise RuntimeError(
                "[FREEZE ERROR] ENFORCE_CLASS_BALANCE=True but some sequences have missing/UNKNOWN group_ID.\n"
                f"Examples (up to 10):\n{bad_examples.to_string(index=False)}")

        unique_groups = sorted(sequences_df["group_ID"].astype(str).unique().tolist())
        print(f"[FREEZE] ENFORCE_CLASS_BALANCE=True --> group_ID classes detected: {unique_groups}")

    # Train/test split
    if not USE_HOLDOUT:
        print("\n[FREEZE] USE_HOLDOUT=False --> X_train will equal X_all; X_test will be empty.")
        train_sequence_indices = sequences_df["sequence_index"].to_numpy()
        test_sequence_indices = np.array([], dtype=int)
        split_mode = "no-holdout"

    else:
        if SPLIT_BY_SUBJECT:
            split_unit = "subject"
        else:
            split_unit = "sequence"
            print(
                "[FREEZE WARN] SPLIT_BY_SUBJECT=False --> train/test split may mix subjects across sets. "
                "This can inflate performance estimates.")

        print(
            f"\n[FREEZE] USE_HOLDOUT=True with HOLDOUT_FRACTION={HOLDOUT_FRACTION:.3f}, "
            f"SPLIT_BY_SUBJECT={SPLIT_BY_SUBJECT}, ENFORCE_CLASS_BALANCE={ENFORCE_CLASS_BALANCE}")

        if SPLIT_BY_SUBJECT:
            unique_subjects = sequences_df["subject_ID"].unique()
            num_subjects = len(unique_subjects)

            if num_subjects < 2:
                print("[FREEZE WARN] Too few subjects to create a holdout split; proceeding without holdout.")
                train_sequence_indices = sequences_df["sequence_index"].to_numpy()
                test_sequence_indices = np.array([], dtype=int)
                split_mode = "fallback-no-holdout-too-few-subjects"
            else:
                if ENFORCE_CLASS_BALANCE:
                    subject_to_group = (
                        sequences_df.groupby("subject_ID")["group_ID"]
                        .agg(lambda s: s.astype(str).mode().iloc[0])
                        .to_dict())

                    subjects_by_group = {}
                    for subj in unique_subjects:
                        g = str(subject_to_group.get(subj))
                        subjects_by_group.setdefault(g, []).append(subj)

                    target_total = max(1, int(round(num_subjects * HOLDOUT_FRACTION)))

                    group_targets = {}
                    remainders = []
                    for g, subjs in subjects_by_group.items():
                        exact = len(subjs) * HOLDOUT_FRACTION
                        base = int(np.floor(exact))
                        group_targets[g] = base
                        remainders.append((g, exact - base))

                    current = sum(group_targets.values())
                    remainders.sort(key=lambda x: x[1], reverse=True)
                    i = 0
                    while current < target_total and i < len(remainders):
                        g = remainders[i][0]
                        group_targets[g] += 1
                        current += 1
                        i += 1

                    for g, subjs in subjects_by_group.items():
                        if len(subjs) >= 2:
                            group_targets[g] = min(group_targets[g], len(subjs) - 1)
                        else:
                            group_targets[g] = min(group_targets[g], len(subjs))

                    test_subjects = set()
                    for g, subjs in subjects_by_group.items():
                        subjs_arr = np.array(subjs, dtype=object)
                        rng.shuffle(subjs_arr)
                        k_take = int(group_targets.get(g, 0))
                        test_subjects.update(subjs_arr[:k_take].tolist())

                    if len(test_subjects) == 0:
                        shuffled = unique_subjects.copy()
                        rng.shuffle(shuffled)
                        test_subjects = {shuffled[0]}

                    train_subjects = set(unique_subjects) - set(test_subjects)

                else:
                    shuffled = unique_subjects.copy()
                    rng.shuffle(shuffled)
                    num_test_subjects = max(1, int(round(num_subjects * HOLDOUT_FRACTION)))
                    num_test_subjects = min(num_test_subjects, num_subjects - 1)
                    test_subjects = set(shuffled[:num_test_subjects])
                    train_subjects = set(shuffled[num_test_subjects:])

                train_mask = sequences_df["subject_ID"].isin(train_subjects)
                test_mask = sequences_df["subject_ID"].isin(test_subjects)

                train_sequence_indices = sequences_df.loc[train_mask, "sequence_index"].to_numpy()
                test_sequence_indices = sequences_df.loc[test_mask, "sequence_index"].to_numpy()
                split_mode = "subject-split"

        else:
            num_sequences = sequences_df.shape[0]
            if num_sequences < 2:
                print("[FREEZE WARN] Too few sequences to create a holdout split; proceeding without holdout.")
                train_sequence_indices = sequences_df["sequence_index"].to_numpy()
                test_sequence_indices = np.array([], dtype=int)
                split_mode = "fallback-no-holdout-too-few-sequences"
            else:
                if ENFORCE_CLASS_BALANCE:
                    groups = sequences_df["group_ID"].astype(str).to_numpy()
                    seq_indices = sequences_df["sequence_index"].to_numpy()
                    unique_groups = np.unique(groups)

                    target_total = max(1, int(round(num_sequences * HOLDOUT_FRACTION)))

                    test_indices = []
                    group_targets = {}
                    remainders = []
                    for g in unique_groups:
                        n_g = int((groups == g).sum())
                        exact = n_g * HOLDOUT_FRACTION
                        base = int(np.floor(exact))
                        group_targets[g] = base
                        remainders.append((g, exact - base))

                    current = sum(group_targets.values())
                    remainders.sort(key=lambda x: x[1], reverse=True)
                    i = 0
                    while current < target_total and i < len(remainders):
                        g = remainders[i][0]
                        group_targets[g] += 1
                        current += 1
                        i += 1

                    for g in unique_groups:
                        candidates = seq_indices[groups == g].copy()
                        rng.shuffle(candidates)
                        k_take = int(group_targets.get(g, 0))
                        test_indices.extend(candidates[:k_take].tolist())

                    if len(test_indices) == 0:
                        shuffled_all = seq_indices.copy()
                        rng.shuffle(shuffled_all)
                        test_indices = [int(shuffled_all[0])]

                    test_sequence_indices = np.array(sorted(set(test_indices)), dtype=int)
                    train_sequence_indices = np.array(
                        [i for i in seq_indices if i not in set(test_sequence_indices)], dtype=int)

                else:
                    all_sequence_indices = sequences_df["sequence_index"].to_numpy()
                    rng.shuffle(all_sequence_indices)
                    num_test_sequences = max(1, int(round(num_sequences * HOLDOUT_FRACTION)))
                    num_test_sequences = min(num_test_sequences, num_sequences - 1)

                    test_sequence_indices = all_sequence_indices[:num_test_sequences]
                    train_sequence_indices = all_sequence_indices[num_test_sequences:]

                split_mode = "sequence-split"

    if USE_HOLDOUT and (test_sequence_indices.size == 0):
        print("[FREEZE WARN] USE_HOLDOUT=True but X_test would be empty; proceeding without holdout.")
        split_mode = f"{split_mode}__effective-no-holdout"
        train_sequence_indices = sequences_df["sequence_index"].to_numpy()

    # Build the three output tables:
    X_all_df = training_df.copy()
    X_train_df, train_sequences_df = build_table_from_sequences(train_sequence_indices)
    X_test_df, test_sequences_df = build_table_from_sequences(test_sequence_indices)

    print("\n[FREEZE] Dataset tables constructed:")
    print(f"   X_all   : {X_all_df.shape}")
    print(f"   X_train : {X_train_df.shape}   (# sequences: {train_sequences_df.shape[0]})")
    print(f"   X_test  : {X_test_df.shape}   (# sequences: {test_sequences_df.shape[0]})")

    # Write 4 outputs:
    x_all_path = DATASET_DIR / "X_all.csv"
    x_train_path = DATASET_DIR / "X_train.csv"
    x_test_path = DATASET_DIR / "X_test.csv"
    provenance_path = DATASET_DIR / "provenance.json"

    X_all_df.to_csv(x_all_path, index=False)
    X_train_df.to_csv(x_train_path, index=False)
    X_test_df.to_csv(x_test_path, index=False)

    # Set provenance payload:
    
    # Core fields first:
    provenance_payload = {
        "dataset_dirname": DATASET_DIR.name,
        "dataset_dir": str(DATASET_DIR),
        "created_files": {
            "X_all": str(x_all_path),
            "X_train": str(x_train_path),
            "X_test": str(x_test_path),
            "provenance": str(provenance_path)},
        "missing_value_handling": HANDLE_MISSING,
        "atlas_family": ATLAS_FAMILY,
        "num_rois": int(NUM_ROIS),
        "epoch_length": EPOCH_LENGTH,
        "epoch_overlap": EPOCH_OVERLAP,
        "use_manifest_filter": bool(USE_MANIFEST_FILTER),
        "overwrite_dataset": bool(OVERWRITE_DATASET),
        "config_snapshot": {
            "ML_CENTRALITY_TYPES": ML_CENTRALITY_TYPES,
            "ML_GRAPH_METRICS": ML_GRAPH_METRICS,
            "STANDARDIZE": bool(STANDARDIZE),
            "CENTER_ONLY": bool(CENTER_ONLY),
            "HANDLE_MISSING": HANDLE_MISSING,
            "USE_HOLDOUT": bool(USE_HOLDOUT),
            "HOLDOUT_FRACTION": float(HOLDOUT_FRACTION) if USE_HOLDOUT else None,
            "SPLIT_BY_SUBJECT": bool(SPLIT_BY_SUBJECT),
            "ENFORCE_CLASS_BALANCE": bool(ENFORCE_CLASS_BALANCE),
            "RANDOM_SEED": RANDOM_SEED},
        "num_rows_all": int(X_all_df.shape[0]),
        "num_rows_train": int(X_train_df.shape[0]),
        "num_rows_test": int(X_test_df.shape[0]),
        "num_features": int(len(feature_columns_final)),
        "num_sequences_all": int(sequences_df.shape[0]),
        "num_sequences_train": int(train_sequences_df.shape[0]),
        "num_sequences_test": int(test_sequences_df.shape[0]),
        "id_columns": id_columns,
        "allowed_metric_names": sorted(list(ALLOWED_METRIC_NAMES)),
        "feature_columns": feature_columns_final,
        "split": {
            "use_holdout": bool(USE_HOLDOUT),
            "holdout_fraction": float(HOLDOUT_FRACTION) if USE_HOLDOUT else None,
            "split_by_subject": bool(SPLIT_BY_SUBJECT),
            "enforce_class_balance": bool(ENFORCE_CLASS_BALANCE),
            "split_mode": split_mode,
            "random_seed_effective": random_seed_effective,
            "train_subjects": sorted(train_sequences_df["subject_ID"].unique().tolist())
            if not train_sequences_df.empty
            else [],
            "test_subjects": sorted(test_sequences_df["subject_ID"].unique().tolist())
            if (test_sequences_df is not None and not test_sequences_df.empty)
            else []},
        "sequence_descriptors": sequence_descriptors,   # <-- single authoritative per-sequence descriptor list (contains length)
        "scaling": scaler_info}     # <-- Scaling parameters (long, but useful for exact reproducibility, e.g. to reverse scaling procedure later if needed)

    with open(provenance_path, "w") as f:
        json.dump(provenance_payload, f, indent=2)

    print("\n[SAVE] Frozen dataset written to:")
    print(f"  - {x_all_path}")
    print(f"  - {x_train_path}")
    print(f"  - {x_test_path}")
    print(f"  - {provenance_path}")

    # Update stable pointer file in OUTPUT_DIR:
    latest_pointer_path = OUTPUT_DIR / "LATEST_DATASET.json"

    latest_payload = {
        "dataset_dir": str(DATASET_DIR),
        "dataset_dirname": DATASET_DIR.name,
        "provenance_path": str(provenance_path),
        "created_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")}

    with open(latest_pointer_path, "w") as f:
        json.dump(latest_payload, f, indent=2)

    print("\n[SAVE] Updated dataset pointer:")
    print(f"  - {latest_pointer_path}")
    print(f"    -> dataset_dir = {latest_payload['dataset_dir']}")

--------